# 01 - Data Understanding
**Project:** DS Job Recommend
**Branch:** `feature/data`
**Vai trò:** Data Engineer / Data Analyst

Mục tiêu: Hiểu cấu trúc, kiểu dữ liệu, mức độ thiếu/trùng lặp, cardinality của toàn bộ 7 bảng dữ liệu tuyển dụng LinkedIn, trước khi làm Data Quality Analysis và Cleaning.

**Nguồn dữ liệu:** LinkedIn Job Postings dataset (Kaggle)
**Các bảng:** `postings`, `companies`, `company_industries`, `company_specialities`, `job_industries`, `job_skills`, `salaries`, `benefits`


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

RAW_DIR = Path('../data/raw')


## 1. Load toàn bộ dataset

`postings.csv` là bảng chính, khá lớn (~500MB) nên load riêng, các bảng còn lại nhỏ hơn nhiều.

In [2]:
# Bảng chính
postings = pd.read_csv(RAW_DIR / 'postings.csv', low_memory=False)

# Các bảng phụ (dimension / bridge tables)
companies = pd.read_csv(RAW_DIR / 'companies.csv')
company_industries = pd.read_csv(RAW_DIR / 'company_industries.csv')
company_specialities = pd.read_csv(RAW_DIR / 'company_specialities.csv')
job_industries = pd.read_csv(RAW_DIR / 'job_industries.csv')
job_skills = pd.read_csv(RAW_DIR / 'job_skills.csv')
salaries = pd.read_csv(RAW_DIR / 'salaries.csv')
benefits = pd.read_csv(RAW_DIR / 'benefits.csv')

datasets = {
    'postings': postings,
    'companies': companies,
    'company_industries': company_industries,
    'company_specialities': company_specialities,
    'job_industries': job_industries,
    'job_skills': job_skills,
    'salaries': salaries,
    'benefits': benefits,
}


## 2. Ghi nhận thông tin dataset gốc (Data Collection)

In [3]:
summary_rows = []
for name, df in datasets.items():
    summary_rows.append({
        'dataset': name,
        'n_records': df.shape[0],
        'n_features': df.shape[1],
        'columns': ', '.join(df.columns)
    })

dataset_overview = pd.DataFrame(summary_rows)
dataset_overview


,dataset,n_records,n_features,columns
0,postings,123849,31,"job_id, company_name, title, description, max_..."
1,companies,24473,10,"company_id, name, description, company_size, s..."
2,company_industries,24375,2,"company_id, industry"
3,company_specialities,169387,2,"company_id, speciality"
4,job_industries,164808,2,"job_id, industry_id"
5,job_skills,213768,2,"job_id, skill_abr"
6,salaries,40785,8,"salary_id, job_id, max_salary, med_salary, min..."
7,benefits,67943,3,"job_id, inferred, type"


**Ghi nhận:**
- Nguồn: LinkedIn Job Postings dataset (Kaggle)
- 8 bảng dữ liệu quan hệ với nhau qua `job_id` / `company_id`
- Không chỉnh sửa file gốc trong `data/raw/` — mọi xử lý đều thao tác trên bản copy trong bộ nhớ, output ghi ra `data/processed/`


## 3. Shape, dtypes, column names — chi tiết từng bảng

In [4]:
def profile_dataset(name, df):
    print(f"===== {name.upper()} =====")
    print('Shape:', df.shape)
    print()
    print('Dtypes:')
    print(df.dtypes)
    print()
    print('Head:')
    display(df.head(3))
    print('\n')

for name, df in datasets.items():
    profile_dataset(name, df)


===== POSTINGS =====
Shape: (123849, 31)

Dtypes:
job_id                          int64
company_name                      str
title                             str
description                       str
max_salary                    float64
pay_period                        str
location                          str
company_id                    float64
views                         float64
med_salary                    float64
min_salary                    float64
formatted_work_type               str
applies                       float64
original_listed_time          float64
remote_allowed                float64
job_posting_url                   str
application_url                   str
application_type                  str
expiry                        float64
closed_time                   float64
formatted_experience_level        str
skills_desc                       str
listed_time                   float64
posting_domain                    str
sponsored                       int64


,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,min_salary,formatted_work_type,applies,original_listed_time,remote_allowed,job_posting_url,application_url,application_type,expiry,closed_time,formatted_experience_level,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,17.0,Full-time,2.0,1.713398e+12,NaN,https://www.linkedin.com/jobs/view/921716/?trk...,NaN,ComplexOnsiteApply,1.715990e+12,NaN,NaN,Requirements: \n\nWe are seeking a College or ...,1.713398e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,38480.0,8540.0,34021.0
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",50.0,HOURLY,"Fort Collins, CO",NaN,1.0,NaN,30.0,Full-time,NaN,1.712858e+12,NaN,https://www.linkedin.com/jobs/view/1829192/?tr...,NaN,ComplexOnsiteApply,1.715450e+12,NaN,NaN,NaN,1.712858e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,83200.0,80521.0,8069.0
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,65000.0,YEARLY,"Cincinnati, OH",64896719.0,8.0,NaN,45000.0,Full-time,NaN,1.713278e+12,NaN,https://www.linkedin.com/jobs/view/10998357/?t...,NaN,ComplexOnsiteApply,1.715870e+12,NaN,NaN,We are currently accepting resumes for FOH - A...,1.713278e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,55000.0,45202.0,39061.0




===== COMPANIES =====
Shape: (24473, 10)

Dtypes:
company_id        int64
name                str
description         str
company_size    float64
state               str
country             str
city                str
zip_code            str
address             str
url                 str
dtype: object

Head:


,company_id,name,description,company_size,state,country,city,zip_code,address,url
0,1009,IBM,"At IBM, we do more than work. We create. We cr...",7.0,NY,US,"Armonk, New York",10504,International Business Machines Corp.,https://www.linkedin.com/company/ibm
1,1016,GE HealthCare,Every day millions of people feel the impact o...,7.0,0,US,Chicago,0,-,https://www.linkedin.com/company/gehealthcare
2,1025,Hewlett Packard Enterprise,Official LinkedIn of Hewlett Packard Enterpris...,7.0,Texas,US,Houston,77389,1701 E Mossy Oaks Rd Spring,https://www.linkedin.com/company/hewlett-packa...




===== COMPANY_INDUSTRIES =====
Shape: (24375, 2)

Dtypes:
company_id    int64
industry        str
dtype: object

Head:


...

,job_id,skill_abr
0,3884428798,MRKT
1,3884428798,PR
2,3884428798,WRT




===== SALARIES =====
Shape: (40785, 8)

Dtypes:
salary_id              int64
job_id                 int64
max_salary           float64
med_salary           float64
min_salary           float64
pay_period               str
currency                 str
compensation_type        str
dtype: object

Head:


,salary_id,job_id,max_salary,med_salary,min_salary,pay_period,currency,compensation_type
0,1,3884428798,NaN,20.0,NaN,HOURLY,USD,BASE_SALARY
1,2,3887470552,25.0,NaN,23.0,HOURLY,USD,BASE_SALARY
2,3,3884431523,120000.0,NaN,100000.0,YEARLY,USD,BASE_SALARY




===== BENEFITS =====
Shape: (67943, 3)

Dtypes:
job_id      int64
inferred    int64
type          str
dtype: object

Head:


,job_id,inferred,type
0,3887473071,0,Medical insurance
1,3887473071,0,Vision insurance
2,3887473071,0,Dental insurance


## 4. Phân loại feature của bảng `postings` theo nhóm nghiệp vụ

In [5]:
feature_groups = {
    'JOB INFORMATION': ['title', 'description', 'skills_desc', 'formatted_experience_level'],
    'SALARY': ['min_salary', 'med_salary', 'max_salary', 'normalized_salary',
               'currency', 'pay_period', 'compensation_type'],
    'LOCATION': ['location', 'zip_code', 'fips'],
    'WORK': ['formatted_work_type', 'work_type', 'remote_allowed'],
    'ENGAGEMENT': ['views', 'applies'],
    'TIME': ['listed_time', 'original_listed_time', 'expiry', 'closed_time'],
    'COMPANY': ['company_id', 'company_name'],
    'OTHER': [],
}

# Cột không thuộc nhóm nào -> OTHER
grouped_cols = {c for cols in feature_groups.values() for c in cols}
feature_groups['OTHER'] = [c for c in postings.columns if c not in grouped_cols]

for group, cols in feature_groups.items():
    existing = [c for c in cols if c in postings.columns]
    missing = [c for c in cols if c not in postings.columns]
    print(f"[{group}] có trong data: {existing}")
    if missing:
        print(f"   -> không tìm thấy: {missing}")


[JOB INFORMATION] có trong data: ['title', 'description', 'skills_desc', 'formatted_experience_level']
[SALARY] có trong data: ['min_salary', 'med_salary', 'max_salary', 'normalized_salary', 'currency', 'pay_period', 'compensation_type']
[LOCATION] có trong data: ['location', 'zip_code', 'fips']
[WORK] có trong data: ['formatted_work_type', 'work_type', 'remote_allowed']
[ENGAGEMENT] có trong data: ['views', 'applies']
[TIME] có trong data: ['listed_time', 'original_listed_time', 'expiry', 'closed_time']
[COMPANY] có trong data: ['company_id', 'company_name']
[OTHER] có trong data: ['job_id', 'job_posting_url', 'application_url', 'application_type', 'posting_domain', 'sponsored']


## 5. Missing values

In [6]:
def missing_report(df):
    miss = df.isna().sum()
    pct = (miss / len(df) * 100).round(2)
    rep = pd.DataFrame({'n_missing': miss, 'pct_missing': pct})
    return rep[rep['n_missing'] > 0].sort_values('pct_missing', ascending=False)

for name, df in datasets.items():
    print(f"--- Missing values: {name} ---")
    rep = missing_report(df)
    if rep.empty:
        print('Không có missing value.')
    else:
        display(rep)
    print()


--- Missing values: postings ---


,n_missing,pct_missing
closed_time,122776,99.13
skills_desc,121410,98.03
med_salary,117569,94.93
remote_allowed,108603,87.69
applies,100529,81.17
min_salary,94056,75.94
max_salary,94056,75.94
pay_period,87776,70.87
compensation_type,87776,70.87
normalized_salary,87776,70.87



--- Missing values: companies ---


,n_missing,pct_missing
company_size,2774,11.33
description,297,1.21
zip_code,28,0.11
address,22,0.09
state,22,0.09
name,1,0.00
city,1,0.00



--- Missing values: company_industries ---
Không có missing value.

--- Missing values: company_specialities ---
Không có missing value.

--- Missing values: job_industries ---
Không có missing value.

--- Missing values: job_skills ---
Không có missing value.

--- Missing values: salaries ---


,n_missing,pct_missing
med_salary,33947,83.23
max_salary,6838,16.77
min_salary,6838,16.77



--- Missing values: benefits ---
Không có missing value.



## 6. Duplicate records

In [7]:
for name, df in datasets.items():
    n_dup = df.duplicated().sum()
    print(f"{name}: {n_dup} dòng trùng lặp hoàn toàn")

# Trùng theo khóa chính nghiệp vụ (không chỉ trùng toàn bộ dòng)
print()
print('postings trùng theo job_id:', postings['job_id'].duplicated().sum())
print('companies trùng theo company_id:', companies['company_id'].duplicated().sum())
print('salaries trùng theo job_id:', salaries['job_id'].duplicated().sum())


postings: 0 dòng trùng lặp hoàn toàn
companies: 0 dòng trùng lặp hoàn toàn
company_industries: 0 dòng trùng lặp hoàn toàn
company_specialities: 0 dòng trùng lặp hoàn toàn
job_industries: 0 dòng trùng lặp hoàn toàn
job_skills: 0 dòng trùng lặp hoàn toàn
salaries: 0 dòng trùng lặp hoàn toàn
benefits: 0 dòng trùng lặp hoàn toàn

postings trùng theo job_id: 0
companies trùng theo company_id: 0
salaries trùng theo job_id: 0


## 7. Unique values & Cardinality (các cột dạng category)

In [8]:
categorical_like_cols = [
    'formatted_work_type', 'work_type', 'formatted_experience_level',
    'pay_period', 'currency', 'compensation_type', 'remote_allowed', 'sponsored'
]

for col in categorical_like_cols:
    if col in postings.columns:
        print(f"--- {col} ---")
        print('n_unique:', postings[col].nunique(dropna=True))
        print(postings[col].value_counts(dropna=False).head(15))
        print()


--- formatted_work_type ---
n_unique: 7
formatted_work_type
Full-time     98814
Contract      12117
Part-time      9696
Temporary      1190
Internship      983
Volunteer       562
Other           487
Name: count, dtype: int64

--- work_type ---
n_unique: 7
work_type
FULL_TIME     98814
CONTRACT      12117
PART_TIME      9696
TEMPORARY      1190
INTERNSHIP      983
VOLUNTEER       562
OTHER           487
Name: count, dtype: int64

--- formatted_experience_level ---
n_unique: 6
formatted_experience_level
Mid-Senior level    41489
Entry level         36708
NaN                 29409
Associate            9826
Director             3746
Internship           1449
Executive            1222
Name: count, dtype: int64

--- pay_period ---
n_unique: 5
pay_period
NaN         87776
YEARLY      20628
HOURLY      14741
MONTHLY       518
WEEKLY        177
BIWEEKLY        9
Name: count, dtype: int64

--- currency ---
n_unique: 6
currency
NaN    87776
USD    36058
EUR        6
CAD        3
BBD        2
AUD

In [9]:
# Cardinality tổng quan cho toàn bộ cột của postings
card = postings.nunique(dropna=True).sort_values(ascending=False)
cardinality_df = pd.DataFrame({
    'n_unique': card,
    'pct_unique': (card / len(postings) * 100).round(2)
})
cardinality_df


,n_unique,pct_unique
job_id,123849,100.00
job_posting_url,123849,100.00
description,107827,87.06
application_url,84800,68.47
title,72521,58.56
original_listed_time,65036,52.51
expiry,54851,44.29
listed_time,53231,42.98
company_id,24474,19.76
company_name,24428,19.72


## 8. Rà soát giá trị bất thường (sơ bộ)

In [10]:
# Salary: giá trị âm hoặc = 0 bất hợp lý
for col in ['min_salary', 'med_salary', 'max_salary', 'normalized_salary']:
    if col in postings.columns:
        n_neg = (postings[col] < 0).sum()
        n_zero = (postings[col] == 0).sum()
        print(f"{col}: {n_neg} giá trị âm, {n_zero} giá trị = 0")

print()
# min_salary > max_salary là dấu hiệu lỗi
if {'min_salary','max_salary'}.issubset(postings.columns):
    bad_range = (postings['min_salary'] > postings['max_salary']).sum()
    print(f'Số dòng min_salary > max_salary: {bad_range}')

print()
# views / applies âm
for col in ['views', 'applies']:
    if col in postings.columns:
        n_neg = (postings[col] < 0).sum()
        print(f"{col}: {n_neg} giá trị âm")


min_salary: 0 giá trị âm, 0 giá trị = 0
med_salary: 0 giá trị âm, 14 giá trị = 0
max_salary: 0 giá trị âm, 0 giá trị = 0
normalized_salary: 0 giá trị âm, 14 giá trị = 0

Số dòng min_salary > max_salary: 0

views: 0 giá trị âm
applies: 0 giá trị âm


In [11]:
# company_size trong companies.csv — kiểm tra range hợp lý (thường là mã 1-7 theo LinkedIn)
print(companies['company_size'].describe())
print()
print(companies['company_size'].value_counts(dropna=False).sort_index())


count    21699.000000
mean         3.349233
std          1.904503
min          1.000000
25%          2.000000
50%          3.000000
75%          5.000000
max          7.000000
Name: company_size, dtype: float64

company_size
1.0    4348
2.0    4956
3.0    3108
4.0    2333
5.0    3918
6.0    1083
7.0    1953
NaN    2774
Name: count, dtype: int64


In [12]:
# Text bị lỗi / rỗng bất thường trong company name, title
print('companies.name rỗng hoặc chỉ có khoảng trắng:',
      companies['name'].astype(str).str.strip().eq('').sum())

if 'title' in postings.columns:
    print('postings.title rỗng hoặc chỉ khoảng trắng:',
          postings['title'].astype(str).str.strip().eq('').sum())


companies.name rỗng hoặc chỉ có khoảng trắng: 0
postings.title rỗng hoặc chỉ khoảng trắng: 0


## 9. Nhận xét sơ bộ (điền sau khi chạy với `postings.csv` đầy đủ)

> Ghi lại tại đây các phát hiện chính sau khi chạy notebook với dữ liệu thật, sẽ dùng làm input cho `02_data_quality_and_cleaning.ipynb`:
> - Cột nào missing nhiều nhất, tỷ lệ bao nhiêu %
> - Cột nào cardinality quá cao (gần bằng n_records) -> có thể là ID/URL, ít giá trị cho model
> - Category nào bị viết không đồng nhất (vd: `Full-time` vs `FULL_TIME`)
> - Giá trị salary bất thường (âm, = 0, min > max) ảnh hưởng bao nhiêu record
